# Neuromorphic-TAME: Deep Space Simulation
**Biological Models of Computation for Deep Space Resilience**

Simulates a Spiking Neural Network (SNN) surviving a Galactic Cosmic Ray (GCR) strike using the TAME morphogenetic repair algorithm inspired by Dr. Michael Levin.

---
**Two parts:**
1. **Grid Animation** — Watch the signal route around the dead zone in real-time.
2. **Output Signal Chart** — The recovery curve: catastrophic signal drop, then autonomous TAME healing.

## Part 1: The SNN Grid Animation
- **Bright nodes** = firing | **Dark** = idle | **Black zone** = dead (GCR damage)

Signal output is measured each frame for the recovery chart in Part 2.

In [ ]:
!pip install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

np.random.seed(42)
GRID_SIZE = 20
NUM_FRAMES = 100
STRIKE_FRAME = 30
REPAIR_FRAME = 60
IDLE, FIRING, DEAD = 0, 1, -1

grid = np.zeros((GRID_SIZE, GRID_SIZE))
weights = np.ones((GRID_SIZE, GRID_SIZE))
output_signal_history = []  # Records throughput each frame for Part 2

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_title('Neuromorphic-TAME Simulation - Healthy Operation', fontsize=12)
ax.axis('off')
im = ax.imshow(grid, cmap='plasma', vmin=-1, vmax=2)

def update(frame):
    global grid, weights
    new_grid = np.copy(grid)
    new_grid[:, 0] = np.random.choice([0, 1], size=GRID_SIZE, p=[0.7, 0.3])
    if frame == STRIKE_FRAME:
        c = GRID_SIZE // 2
        new_grid[c-3:c+3, c-3:c+3] = DEAD
        ax.set_title('CRITICAL FAULT: Galactic Cosmic Ray Strike', fontsize=12, color='#ff4444')
    if frame > STRIKE_FRAME:
        c = GRID_SIZE // 2
        new_grid[c-3:c+3, c-3:c+3] = DEAD
    if frame == REPAIR_FRAME:
        c = GRID_SIZE // 2
        weights[c-4:c+4, c-4:c+4] = 2.5
        ax.set_title('TAME ACTIVATED: Morphogenetic Plasticity Repairing Network', fontsize=11, color='#00ffcc')
    if frame > REPAIR_FRAME + 10:
        ax.set_title('SYSTEM RECOVERED: Signal Rerouted via TAME Plasticity', fontsize=11, color='#44ff88')
    for i in range(GRID_SIZE):
        for j in range(1, GRID_SIZE):
            if new_grid[i, j] == DEAD:
                continue
            firing_neighbors = 0
            radius = 3 if weights[i, j] > 1.0 else 1
            for di in range(-radius, radius + 1):
                for dj in range(-radius, 0):
                    ni, nj = i + di, j + dj
                    if 0 <= ni < GRID_SIZE and 0 <= nj < GRID_SIZE:
                        if grid[ni, nj] == FIRING:
                            firing_neighbors += weights[i, j]
            if firing_neighbors > 0 and new_grid[i, j] != DEAD:
                new_grid[i, j] = FIRING if np.random.rand() < 0.6 + (0.1 * weights[i, j]) else IDLE
            elif new_grid[i, j] != DEAD:
                new_grid[i, j] = IDLE
    grid = new_grid
    output_signal_history.append(int(np.sum(grid[:, -1] == FIRING)))
    vis_grid = np.copy(grid).astype(float)
    vis_grid[vis_grid == IDLE] = 0.2
    vis_grid[vis_grid == FIRING] = 2.0
    vis_grid[vis_grid == DEAD] = -1.0
    im.set_array(vis_grid)
    return [im]

anim = animation.FuncAnimation(fig, update, frames=NUM_FRAMES, interval=100, blit=True)
plt.close()
HTML(anim.to_jshtml())

---
## Part 2: Output Signal Chart - The TAME Recovery Curve

Plots signal throughput at the output edge across all 100 frames.

- **Green zone (0-30):** Healthy baseline
- **Red zone (30-60):** Post-GCR signal collapse
- **Cyan zone (60-100):** TAME activates - signal recovers

In [ ]:
if len(output_signal_history) == 0:
    print('No data yet. Run the animation cell above first, then re-run this cell.')
else:
    frames = np.arange(len(output_signal_history))
    def rolling_avg(arr, window=5):
        return np.convolve(arr, np.ones(window)/window, mode='same')
    smoothed = rolling_avg(output_signal_history)
    baseline = np.mean(output_signal_history[:STRIKE_FRAME])

    plt.style.use('dark_background')
    fig2, ax2 = plt.subplots(figsize=(12, 6))

    # Raw signal - silver/grey dashed so it reads clearly on dark background
    ax2.plot(frames, output_signal_history, color='#aaaaaa', alpha=0.7, linewidth=1.2,
             linestyle='--', label='Raw Output Signal')
    # Smoothed hero line
    ax2.plot(frames, smoothed, color='#00ffcc', linewidth=2.5, label='Signal Throughput (smoothed)')
    ax2.axhline(y=baseline, color='#ffffff', linestyle='--', alpha=0.5,
                label=f'Pre-Strike Baseline ({baseline:.1f} nodes)')

    ax2.axvline(x=STRIKE_FRAME, color='#ff4444', linestyle=':', linewidth=2, label='GCR Strike (Frame 30)')
    ax2.axvline(x=REPAIR_FRAME, color='#ffaa00', linestyle=':', linewidth=2, label='TAME Activated (Frame 60)')

    ax2.axvspan(0, STRIKE_FRAME, alpha=0.05, color='green')
    ax2.axvspan(STRIKE_FRAME, REPAIR_FRAME, alpha=0.08, color='red')
    ax2.axvspan(REPAIR_FRAME, NUM_FRAMES, alpha=0.05, color='cyan')

    ax2.annotate('Catastrophic\nSignal Loss',
                 xy=(STRIKE_FRAME + 2, smoothed[STRIKE_FRAME + 2]),
                 xytext=(STRIKE_FRAME + 6, baseline * 0.85),
                 arrowprops=dict(facecolor='#ff4444', arrowstyle='->', lw=1.5),
                 color='#ff4444', fontsize=10)
    ax2.annotate('Autonomous\nTAME Recovery',
                 xy=(REPAIR_FRAME + 8, smoothed[min(REPAIR_FRAME + 8, len(smoothed)-1)]),
                 xytext=(REPAIR_FRAME + 14, baseline * 0.35),
                 arrowprops=dict(facecolor='#00ffcc', arrowstyle='->', lw=1.5),
                 color='#00ffcc', fontsize=10)

    ax2.set_title('Neuromorphic SNN: Signal Throughput Under GCR Strike & TAME Repair',
                  fontsize=13, loc='left', pad=15)
    ax2.set_xlabel('Mission Timeline (Simulation Frames)', fontsize=11)
    ax2.set_ylabel('Output Nodes Firing (Signal Throughput)', fontsize=11)
    ax2.set_ylim(bottom=0)
    # Legend with bright white text on dark panel - fully readable
    ax2.legend(loc='lower right', fontsize=9,
               facecolor='#1a1a2e', edgecolor='#444444', labelcolor='white')
    ax2.grid(True, alpha=0.1)
    plt.tight_layout()
    plt.savefig('neuromorphic_tame_output_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved as neuromorphic_tame_output_chart.png')